# 🎯 Phase 3: Milestone Exam Solutions

> **Data Engineering & Web Development**
>
> This notebook contains comprehensive solutions for all four Phase Milestone Exam questions.
> Each solution demonstrates the synthesis of concepts from Days 25-36.

---

## Question 1: ETL Pipeline — Stock Data Ingestion

**Combines**: APIs (Day 31), Pandas (Day 25-26), SQLite (Day 30), Matplotlib (Day 27-28)

**Scenario**: Build a full **Extract → Transform → Load** pipeline that:
1. **Extracts** stock price data (simulated API response)
2. **Transforms** it — calculates daily returns, moving averages, volume flags
3. **Loads** it into a SQLite database
4. **Visualizes** the results with matplotlib

> ⚠️ `requests` is not available in Pyodide. We simulate the API response with inline JSON data.

In [ ]:
import json
import sqlite3
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Pyodide
import matplotlib.pyplot as plt

# === EXTRACT: Simulated API Response ===
# In production: response = requests.get(f'https://api.stock.com/daily/{symbol}')
# In production: data = response.json()

STOCK_API_RESPONSE = [
    {"date": "2024-01-02", "symbol": "AAPL", "open": 185.5, "close": 186.2, "high": 187.0, "low": 184.8, "volume": 45000000},
    {"date": "2024-01-03", "symbol": "AAPL", "open": 186.0, "close": 184.5, "high": 186.5, "low": 183.9, "volume": 52000000},
    {"date": "2024-01-04", "symbol": "AAPL", "open": 184.2, "close": 185.8, "high": 186.1, "low": 183.5, "volume": 48000000},
    {"date": "2024-01-05", "symbol": "AAPL", "open": 186.0, "close": 187.3, "high": 188.0, "low": 185.5, "volume": 55000000},
    {"date": "2024-01-08", "symbol": "AAPL", "open": 187.0, "close": 188.5, "high": 189.2, "low": 186.5, "volume": 42000000},
    {"date": "2024-01-09", "symbol": "AAPL", "open": 188.2, "close": 186.8, "high": 188.8, "low": 186.0, "volume": 50000000},
    {"date": "2024-01-10", "symbol": "AAPL", "open": 187.0, "close": 189.1, "high": 189.5, "low": 186.8, "volume": 47000000},
    {"date": "2024-01-02", "symbol": "GOOGL", "open": 140.2, "close": 141.5, "high": 142.0, "low": 139.8, "volume": 28000000},
    {"date": "2024-01-03", "symbol": "GOOGL", "open": 141.0, "close": 139.8, "high": 141.5, "low": 138.5, "volume": 32000000},
    {"date": "2024-01-04", "symbol": "GOOGL", "open": 140.0, "close": 142.1, "high": 143.0, "low": 139.5, "volume": 30000000},
    {"date": "2024-01-05", "symbol": "GOOGL", "open": 142.5, "close": 143.8, "high": 144.2, "low": 141.8, "volume": 35000000},
    {"date": "2024-01-08", "symbol": "GOOGL", "open": 143.5, "close": 144.2, "high": 145.0, "low": 143.0, "volume": 26000000},
    {"date": "2024-01-09", "symbol": "GOOGL", "open": 144.0, "close": 142.5, "high": 144.5, "low": 141.8, "volume": 33000000},
    {"date": "2024-01-10", "symbol": "GOOGL", "open": 143.0, "close": 145.0, "high": 145.5, "low": 142.5, "volume": 29000000}
]

print(f"Extracted {len(STOCK_API_RESPONSE)} records from simulated API")

In [ ]:
# === TRANSFORM ===
def transform_stock_data(raw_data):
    """
    Transform raw stock data into analysis-ready DataFrame.

    Adds:
    - daily_return:  (close - open) / open * 100
    - price_range:   high - low
    - sma_3:         3-day simple moving average per symbol
    - high_volume:   True if volume > median for that symbol

    Args:
        raw_data: List of dicts from API

    Returns:
        pd.DataFrame: Transformed data
    """
    df = pd.DataFrame(raw_data)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

    # Calculate derived metrics
    df["daily_return"] = ((df["close"] - df["open"]) / df["open"] * 100).round(4)
    df["price_range"] = (df["high"] - df["low"]).round(2)

    # 3-day simple moving average (per symbol)
    df["sma_3"] = (
        df.groupby("symbol")["close"]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
        .round(2)
    )

    # High volume flag (above median for that symbol)
    df["high_volume"] = df.groupby("symbol")["volume"].transform(
        lambda x: x > x.median()
    )

    return df


transformed = transform_stock_data(STOCK_API_RESPONSE)
print("Transformed Data:")
print(transformed[["date", "symbol", "close", "daily_return", "sma_3", "high_volume"]].to_string(index=False))

In [ ]:
# === LOAD (into SQLite) ===
def load_to_db(df, db_name=":memory:"):
    """
    Load DataFrame into a SQLite database.

    Creates a 'stock_prices' table and loads data using
    INSERT OR REPLACE for idempotency.

    Args:
        df: Transformed DataFrame
        db_name: Database file path (":memory:" for in-memory)

    Returns:
        sqlite3.Connection: Active database connection
    """
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Create table with proper schema
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS stock_prices (
            date TEXT,
            symbol TEXT,
            open REAL,
            close REAL,
            high REAL,
            low REAL,
            volume INTEGER,
            daily_return REAL,
            price_range REAL,
            sma_3 REAL,
            high_volume BOOLEAN,
            PRIMARY KEY (date, symbol)
        )
    """)

    # Load data (Pandas to_sql with replace)
    df_to_load = df.copy()
    df_to_load["date"] = df_to_load["date"].dt.strftime("%Y-%m-%d")
    df_to_load.to_sql("stock_prices", conn, if_exists="replace", index=False)

    print(f"Loaded {len(df)} records into stock_prices table")
    return conn


conn = load_to_db(transformed)

# Verify with a query
result = pd.read_sql("SELECT symbol, COUNT(*) as days, ROUND(AVG(daily_return), 4) as avg_return FROM stock_prices GROUP BY symbol", conn)
print("\nVerification Query:")
print(result.to_string(index=False))

In [ ]:
# === VISUALIZE ===
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

for symbol in transformed["symbol"].unique():
    data = transformed[transformed["symbol"] == symbol]
    axes[0].plot(data["date"], data["close"], marker="o", label=f"{symbol} Close")
    axes[0].plot(data["date"], data["sma_3"], linestyle="--", alpha=0.7, label=f"{symbol} SMA(3)")

axes[0].set_title("Stock Prices with 3-Day Moving Average")
axes[0].set_ylabel("Price ($)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for symbol in transformed["symbol"].unique():
    data = transformed[transformed["symbol"] == symbol]
    axes[1].bar(data["date"].astype(str), data["daily_return"], alpha=0.7, label=symbol)

axes[1].set_title("Daily Returns (%)")
axes[1].set_ylabel("Return (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color="black", linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("stock_analysis.png", dpi=100)
print("Chart saved as stock_analysis.png")
plt.show()

conn.close()

---

## Question 2: Multi-Page Web Scraper

**Combines**: Web Scraping (Day 29), Regex (Day 17), Data Cleaning (Day 25)

**Scenario**: Build a scraper that parses product listings from an e-commerce page.

> ⚠️ `requests` and `BeautifulSoup` are not available in Pyodide.
> We simulate the HTML parsing logic using inline HTML strings. The scraping pattern (parse → clean → aggregate) is fully demonstrated.

In [ ]:
import re
from collections import defaultdict

# Simulated HTML pages (in production: requests.get(url).text)
SIMULATED_PAGES = [
    """
    <div class="product">
        <h3>Wireless Mouse</h3>
        <span class="price">$29.99</span>
        <span class="category">Electronics</span>
        <span class="rating">4.5/5</span>
        <span class="stock">In Stock</span>
    </div>
    <div class="product">
        <h3>Standing Desk</h3>
        <span class="price">$499.99</span>
        <span class="category">Furniture</span>
        <span class="rating">4.8/5</span>
        <span class="stock">In Stock</span>
    </div>
    <div class="product">
        <h3>USB-C Hub</h3>
        <span class="price">$39.95</span>
        <span class="category">Electronics</span>
        <span class="rating">4.2/5</span>
        <span class="stock">Out of Stock</span>
    </div>
    """,
    """
    <div class="product">
        <h3>Ergonomic Chair</h3>
        <span class="price">$899.00</span>
        <span class="category">Furniture</span>
        <span class="rating">4.7/5</span>
        <span class="stock">In Stock</span>
    </div>
    <div class="product">
        <h3>Noise Cancelling Headphones</h3>
        <span class="price">$249.99</span>
        <span class="category">Electronics</span>
        <span class="rating">4.8/5</span>
        <span class="stock">In Stock</span>
    </div>
    <div class="product">
        <h3>Desk Lamp</h3>
        <span class="price">$34.50</span>
        <span class="category">Furniture</span>
        <span class="rating">4.1/5</span>
        <span class="stock">Out of Stock</span>
    </div>
    """
]

print(f"Simulated {len(SIMULATED_PAGES)} HTML pages")

In [ ]:
def parse_product_html(html):
    """
    Parse product listings from HTML using regex.

    In production, you'd use BeautifulSoup:
        soup = BeautifulSoup(html, 'html.parser')
        products = soup.find_all('div', class_='product')

    Args:
        html: Raw HTML string

    Returns:
        list[dict]: List of parsed product dictionaries
    """
    products = []
    # Split by product blocks
    blocks = re.findall(r'<div class="product">(.*?)</div>', html, re.DOTALL)

    for block in blocks:
        name = re.search(r'<h3>(.*?)</h3>', block)
        price = re.search(r'class="price">\$(\d+\.?\d*)', block)
        category = re.search(r'class="category">(.*?)</span>', block)
        rating = re.search(r'class="rating">(\d+\.?\d*)/5', block)
        stock = re.search(r'class="stock">(.*?)</span>', block)

        if name and price:
            products.append({
                "name": name.group(1).strip(),
                "price": float(price.group(1)),
                "category": category.group(1).strip() if category else "Unknown",
                "rating": float(rating.group(1)) if rating else None,
                "in_stock": stock.group(1).strip() == "In Stock" if stock else False,
            })
    return products


def scrape_all_pages(pages):
    """
    Scrape product data from multiple pages.

    In production, this would handle:
    - Pagination: follow 'next page' links
    - Rate limiting: time.sleep(1) between requests
    - Error handling: retry on 429/500

    Args:
        pages: List of HTML strings (simulated pages)

    Returns:
        list[dict]: All products across all pages
    """
    all_products = []
    for i, page_html in enumerate(pages, 1):
        products = parse_product_html(page_html)
        all_products.extend(products)
        print(f"  Page {i}: Found {len(products)} products")
        # In production: time.sleep(1)  # Be polite
    return all_products

In [ ]:
# Test the Scraper
print("=" * 50)
print("WEB SCRAPER TEST")
print("=" * 50)

products = scrape_all_pages(SIMULATED_PAGES)

print(f"\nTotal products scraped: {len(products)}")
print(f"In stock: {sum(1 for p in products if p['in_stock'])}")

# Category statistics
print("\n📊 Category Statistics:")
stats = defaultdict(lambda: {"count": 0, "total_price": 0, "prices": []})
for p in products:
    cat = p["category"]
    stats[cat]["count"] += 1
    stats[cat]["total_price"] += p["price"]
    stats[cat]["prices"].append(p["price"])

for cat, data in stats.items():
    avg = data["total_price"] / data["count"]
    mn, mx = min(data["prices"]), max(data["prices"])
    print(f"  {cat}: {data['count']} products, avg ${avg:.2f}, range ${mn:.2f}-${mx:.2f}")

# Top rated products
print("\n🏆 Top Rated:")
for p in sorted(products, key=lambda x: x['rating'] or 0, reverse=True)[:3]:
    stock = "✅" if p["in_stock"] else "❌"
    print(f"  {stock} {p['name']} — ${p['price']:.2f} ({p['rating']}/5)")

---

## Question 3: REST API with Database

**Combines**: SQLite (Day 30), APIs (Day 31), HTTP Methods (Day 33)

**Scenario**: Implement CRUD operations for a Task Manager API backed by SQLite.

> ⚠️ `FastAPI`/`Flask` are not available in Pyodide. We demonstrate the full request-handling logic (route → validate → query → respond) as functions. In production, you'd wrap each function with `@app.get('/tasks')`, `@app.post('/tasks')`, etc.

In [ ]:
import sqlite3
import json
from datetime import datetime


class TaskManagerAPI:
    """
    REST API logic for a Task Manager, backed by SQLite.

    In production, each method would be a route handler:
        @app.get('/tasks')      -> list_tasks()
        @app.post('/tasks')     -> create_task()
        @app.put('/tasks/{id}') -> update_task()
        @app.delete('/tasks/{id}') -> delete_task()
    """

    def __init__(self, db_name=":memory:"):
        self.conn = sqlite3.connect(db_name)
        self.conn.row_factory = sqlite3.Row
        self._create_tables()

    def _create_tables(self):
        """Initialize the database schema."""
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                description TEXT DEFAULT '',
                status TEXT DEFAULT 'pending' CHECK(status IN ('pending', 'in_progress', 'done')),
                priority INTEGER DEFAULT 3 CHECK(priority BETWEEN 1 AND 5),
                created_at TEXT DEFAULT (datetime('now')),
                updated_at TEXT DEFAULT (datetime('now'))
            )
        """)
        self.conn.commit()

    def _response(self, status_code, data=None, error=None):
        """Build a standardized API response."""
        resp = {"status": status_code}
        if data is not None:
            resp["data"] = data
        if error:
            resp["error"] = error
        return resp

    # GET /tasks
    def list_tasks(self, status_filter=None):
        """List all tasks, optionally filtered by status."""
        query = "SELECT * FROM tasks"
        params = []
        if status_filter:
            query += " WHERE status = ?"
            params.append(status_filter)
        query += " ORDER BY priority ASC, created_at DESC"

        rows = self.conn.execute(query, params).fetchall()
        tasks = [dict(row) for row in rows]
        return self._response(200, tasks)

    # POST /tasks
    def create_task(self, payload):
        """Create a new task from request body."""
        title = payload.get("title", "").strip()
        if not title:
            return self._response(400, error="Title is required")

        description = payload.get("description", "")
        priority = payload.get("priority", 3)

        if not (1 <= priority <= 5):
            return self._response(400, error="Priority must be 1-5")

        cursor = self.conn.execute(
            "INSERT INTO tasks (title, description, priority) VALUES (?, ?, ?)",
            (title, description, priority)
        )
        self.conn.commit()

        new_task = dict(self.conn.execute("SELECT * FROM tasks WHERE id = ?", (cursor.lastrowid,)).fetchone())
        return self._response(201, new_task)

    # PUT /tasks/{id}
    def update_task(self, task_id, payload):
        """Update an existing task."""
        existing = self.conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
        if not existing:
            return self._response(404, error=f"Task {task_id} not found")

        updates = []
        params = []
        for field in ["title", "description", "status", "priority"]:
            if field in payload:
                updates.append(f"{field} = ?")
                params.append(payload[field])

        if not updates:
            return self._response(400, error="No fields to update")

        updates.append("updated_at = datetime('now')")
        params.append(task_id)

        self.conn.execute(f"UPDATE tasks SET {', '.join(updates)} WHERE id = ?", params)
        self.conn.commit()

        updated = dict(self.conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone())
        return self._response(200, updated)

    # DELETE /tasks/{id}
    def delete_task(self, task_id):
        """Delete a task by ID."""
        existing = self.conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
        if not existing:
            return self._response(404, error=f"Task {task_id} not found")

        self.conn.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
        self.conn.commit()
        return self._response(200, {"deleted": task_id})

In [ ]:
# Test the REST API
print("=" * 50)
print("REST API TEST (CRUD Operations)")
print("=" * 50)

api = TaskManagerAPI()

# CREATE
print("\n📝 POST /tasks")
r1 = api.create_task({"title": "Build ETL Pipeline", "description": "Extract, transform, load stock data", "priority": 1})
print(f"  {r1['status']}: {r1['data']['title']} (id={r1['data']['id']})")

r2 = api.create_task({"title": "Write Unit Tests", "priority": 2})
print(f"  {r2['status']}: {r2['data']['title']} (id={r2['data']['id']})")

r3 = api.create_task({"title": "Deploy to Production", "priority": 3})
print(f"  {r3['status']}: {r3['data']['title']} (id={r3['data']['id']})")

# Error case
r_err = api.create_task({"title": ""})
print(f"  {r_err['status']}: {r_err.get('error', 'no error')}")

# READ
print("\n📋 GET /tasks")
r = api.list_tasks()
for task in r["data"]:
    print(f"  [{task['status']}] P{task['priority']}: {task['title']}")

# UPDATE
print("\n✏️ PUT /tasks/1")
r = api.update_task(1, {"status": "in_progress"})
print(f"  {r['status']}: {r['data']['title']} → {r['data']['status']}")

# DELETE
print("\n🗑️ DELETE /tasks/3")
r = api.delete_task(3)
print(f"  {r['status']}: deleted task {r['data']['deleted']}")

# Final state
print("\n📋 Final State:")
r = api.list_tasks()
for task in r["data"]:
    print(f"  #{task['id']} [{task['status']}] P{task['priority']}: {task['title']}")

---

## Question 4: Flask Analytics Dashboard (Logic)

**Combines**: Flask (Day 34), SQLite (Day 30), Matplotlib (Day 27), Jinja (Day 34)

**Scenario**: Build the *data backend* for a web analytics dashboard — SQL queries for KPIs, data aggregation, and chart generation.

> ⚠️ Flask is not available in Pyodide. We demonstrate the backend logic:
> - Database queries that would power each dashboard widget
> - Data aggregation for chart rendering
> - The Jinja template context each route would produce

In [ ]:
import sqlite3
import random

# Seed a database with synthetic analytics data
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.executescript("""
    CREATE TABLE page_views (
        id INTEGER PRIMARY KEY,
        page TEXT,
        user_id INTEGER,
        timestamp TEXT,
        duration_seconds INTEGER,
        referrer TEXT
    );

    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        signup_date TEXT,
        plan TEXT CHECK(plan IN ('free', 'pro', 'enterprise'))
    );
""")

# Seed users
random.seed(42)
plans = ["free", "free", "free", "pro", "pro", "enterprise"]
for i in range(1, 51):
    month = random.randint(1, 12)
    cursor.execute(
        "INSERT INTO users (id, name, signup_date, plan) VALUES (?, ?, ?, ?)",
        (i, f"User_{i}", f"2024-{month:02d}-{random.randint(1,28):02d}", random.choice(plans))
    )

# Seed page views
pages = ["/home", "/products", "/about", "/pricing", "/docs", "/blog"]
referrers = ["google", "direct", "twitter", "linkedin", "github"]
for i in range(1, 501):
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    hour = random.randint(8, 22)
    cursor.execute(
        "INSERT INTO page_views (page, user_id, timestamp, duration_seconds, referrer) VALUES (?, ?, ?, ?, ?)",
        (random.choice(pages), random.randint(1, 50),
         f"2024-{month:02d}-{day:02d} {hour:02d}:{random.randint(0,59):02d}:00",
         random.randint(5, 300), random.choice(referrers))
    )

conn.commit()
print("Database seeded: 50 users, 500 page views")

In [ ]:
# === Dashboard Backend: KPI Queries ===
# These are the exact queries that would power each widget

def get_dashboard_context(conn):
    """
    Generate the full context dictionary for the dashboard template.

    In Flask:
        @app.route('/dashboard')
        def dashboard():
            ctx = get_dashboard_context(db.connection)
            return render_template('dashboard.html', **ctx)

    Returns:
        dict: All data needed by the Jinja template
    """
    cur = conn.cursor()

    # KPI 1: Total Views & Unique Visitors
    kpi = cur.execute("""
        SELECT
            COUNT(*) as total_views,
            COUNT(DISTINCT user_id) as unique_visitors,
            ROUND(AVG(duration_seconds), 1) as avg_duration
        FROM page_views
    """).fetchone()

    # KPI 2: Top Pages
    top_pages = cur.execute("""
        SELECT page,
               COUNT(*) as views,
               ROUND(AVG(duration_seconds), 0) as avg_time
        FROM page_views
        GROUP BY page
        ORDER BY views DESC
    """).fetchall()

    # KPI 3: Traffic by Referrer
    referrer_data = cur.execute("""
        SELECT referrer,
               COUNT(*) as visits,
               ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM page_views), 1) as pct
        FROM page_views
        GROUP BY referrer
        ORDER BY visits DESC
    """).fetchall()

    # KPI 4: Monthly Trend
    monthly_trend = cur.execute("""
        SELECT
            substr(timestamp, 1, 7) as month,
            COUNT(*) as views,
            COUNT(DISTINCT user_id) as unique_users
        FROM page_views
        GROUP BY month
        ORDER BY month
    """).fetchall()

    # KPI 5: User Plan Distribution
    plan_dist = cur.execute("""
        SELECT plan, COUNT(*) as count
        FROM users
        GROUP BY plan
        ORDER BY count DESC
    """).fetchall()

    return {
        "total_views": kpi[0],
        "unique_visitors": kpi[1],
        "avg_duration": kpi[2],
        "top_pages": [(r[0], r[1], r[2]) for r in top_pages],
        "referrers": [(r[0], r[1], r[2]) for r in referrer_data],
        "monthly_trend": [(r[0], r[1], r[2]) for r in monthly_trend],
        "plan_distribution": [(r[0], r[1]) for r in plan_dist],
    }

In [ ]:
# Test the Dashboard Backend
print("=" * 50)
print("ANALYTICS DASHBOARD BACKEND")
print("=" * 50)

ctx = get_dashboard_context(conn)

print(f"\n📊 KPIs:")
print(f"  Total Page Views:   {ctx['total_views']}")
print(f"  Unique Visitors:    {ctx['unique_visitors']}")
print(f"  Avg Duration:       {ctx['avg_duration']}s")

print(f"\n📄 Top Pages:")
for page, views, avg_time in ctx['top_pages']:
    bar = '█' * (views // 10)
    print(f"  {page:15s} {views:4d} views ({avg_time}s avg) {bar}")

print(f"\n🔗 Traffic Sources:")
for ref, visits, pct in ctx['referrers']:
    bar = '█' * int(pct / 2)
    print(f"  {ref:12s} {visits:4d} ({pct}%) {bar}")

print(f"\n📈 Monthly Trend:")
for month, views, users in ctx['monthly_trend']:
    bar = '█' * (views // 5)
    print(f"  {month}:  {views:4d} views, {users:3d} users {bar}")

print(f"\n👥 User Plans:")
for plan, count in ctx['plan_distribution']:
    print(f"  {plan:12s} {count}")

conn.close()

---

## 🎓 Summary

This notebook demonstrated solutions to all four Phase 3 Milestone Exam questions:

1. **ETL Pipeline**: Full Extract → Transform → Load workflow with pandas, sqlite3, and matplotlib
2. **Multi-Page Web Scraper**: HTML parsing with regex, data cleaning, and category statistics
3. **REST API with Database**: Complete CRUD operations with SQLite, validation, and error handling
4. **Flask Analytics Dashboard**: SQL-powered KPI queries and data aggregation for dashboard widgets

Each solution follows best practices including:
- Production-ready patterns (idempotent loads, standardized API responses)
- Graceful degradation for browser runtime (mocked external dependencies)
- Clear documentation of what would differ in production